<a href="https://colab.research.google.com/github/eriksetiawan018-Maverick/Erik-Setiawan-UAS-ADKP-Electrical-Safety-/blob/main/Erik%20Setiawan-Uas%20ADKP%20Electrical%20Safety%20Monte%20Carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Erik Setiawan-Uas ADKP Electrical Safety Monte Carlo

## Prediksi Skor Risiko Keselamatan Listrik + Simulasi Monte Carlo

- **Penulis:** Erik Setiawan (NPM: 2506564102)
- **Program Studi:** S2 Teknik Kimia — Manajemen Keselamatan Proses
- **Mata Kuliah:** Analisis Data Keselamatan Proses (UAS)

Notebook ini membangun **model regresi** untuk memprediksi `Electrical_Safety_Risk_Score` (skor risiko keselamatan listrik, skala 0–100) dari data pekerjaan listrik di data center, lalu memakai model terbaik di dalam **simulasi Monte Carlo** untuk menjawab pertanyaan *what-if*, misalnya:

- Berapa persen pekerjaan yang masuk kategori risiko **Medium/High** dalam kondisi sekarang?
- Apa yang terjadi kalau semua pekerjaan dilakukan saat **shift malam**?
- Seberapa besar risiko turun kalau **pelanggaran LOTO dihilangkan** atau **kelelahan pekerja dikurangi**?

**Alur:**
1. Memuat data
2. Eksplorasi data (EDA)
3. Pembersihan & pemilihan fitur (termasuk membuang kolom yang "bocor")
4. Melatih dan membandingkan model regresi
5. Faktor yang paling berpengaruh
6. Simulasi Monte Carlo: (6.1) skenario what-if dan (6.2) analisis ketidakpastian probabilistik + tornado chart
7. Kesimpulan
8. Referensi

## 1. Memuat data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
# Dataset: electrical_safety_5000.csv (Kaggle)
# Di Google Colab: upload file CSV lewat panel Files, atau jalankan sel ini dan pilih file saat diminta.
DATA_PATH = "electrical_safety_5000.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files  # hanya ada di Colab
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError("Taruh electrical_safety_5000.csv di folder yang sama dengan notebook ini.")

df = pd.read_csv(DATA_PATH)
print("Ukuran data:", df.shape)
df.head()

## 2. Eksplorasi data (EDA)

In [ ]:
TARGET = "Electrical_Safety_Risk_Score"

print("Jumlah baris duplikat:", df.duplicated().sum())
missing = df.isna().sum()
print("Kolom dengan data kosong:")
print(missing[missing > 0])
print()
print(df[TARGET].describe())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df[TARGET], bins=40, kde=True, ax=ax[0], color="steelblue")
ax[0].axvline(40, color="orange", ls="--", label="batas Medium (40)")
ax[0].axvline(65, color="red", ls="--", label="batas High (65)")
ax[0].set_title("Distribusi skor risiko keselamatan listrik")
ax[0].legend()

df["Risk_Level"].value_counts().reindex(["Low", "Medium", "High"]).plot.bar(ax=ax[1], color=["seagreen", "orange", "red"])
ax[1].set_title("Jumlah data per Risk_Level")
plt.tight_layout(); plt.show()

# Risk_Level ternyata hanya potongan dari skor: <40 Low, 40–65 Medium, >65 High
df.groupby("Risk_Level")[TARGET].agg(["min", "max", "count"])

In [ ]:
# Korelasi kolom numerik dengan target
num_corr = df.select_dtypes("number").corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
num_corr.head(15)

**Temuan penting dari EDA:**
- Data bersih: tidak ada duplikat; kolom kosong hanya `Injury_Severity` dan `Electrical_Incident_Severity` (kosong karena memang tidak ada cedera/insiden).
- `Predictive_Risk_Score` berkorelasi **1,0** dan `Probability_of_Incident` **0,97** dengan target. Kolom ini kemungkinan dihitung dari target (atau sebaliknya), jadi kalau dipakai sebagai fitur, model akan "nyontek" (*data leakage*). Keduanya **dibuang**.
- `Risk_Level` hanyalah hasil potong skor (Low < 40, Medium 40–65, High > 65), jadi juga dibuang dari fitur, tapi batasnya dipakai lagi di simulasi Monte Carlo.
- Distribusi kategori sangat merata, sehingga dataset ini kemungkinan besar **data sintetis**. Hasil model menggambarkan pola di data ini, bukan kondisi data center nyata.

## 3. Pembersihan & pemilihan fitur

In [ ]:
# (a) Kolom yang bocor / turunan langsung dari target atau hasil insiden (terjadi SETELAH kejadian)
leak_cols = [
    TARGET, "Predictive_Risk_Score", "Probability_of_Incident", "Risk_Level",
    "Electrical_Incident_Occurred", "Electrical_Incident_Severity", "Incident_Occurred_Target",
    "Near_Miss_Flag", "Injury_Occurred", "Injury_Severity", "Medical_Treatment_Required",
    "Lost_Time_Incident", "Fatality_Flag", "Equipment_Damage", "Service_Impact",
    "Incident_Type", "Incident_Category", "Root_Cause", "Corrective_Action",
]
# (b) Kolom ID, tanggal, jam, dan teks bebas (tidak bermakna sebagai fitur)
id_cols = [c for c in df.columns if c.endswith("_ID") or c in ["Serial_Number", "Model", "Tile", "Rack"]]
date_cols = [c for c in df.columns if c.endswith("_Date") or c.endswith("_Time")]
text_cols = ["Incident_Description", "Investigation_Notes", "Corrective_Action_Text", "Lessons_Learned", "Root_Cause_Text"]
# (c) Kolom yang nilainya sama semua
const_cols = [c for c in df.columns if df[c].nunique() <= 1]
# (d) Indeks gabungan yang dihitung dari kolom mentah lain, misalnya:
#     Fatigue_Index = Fatigue_Score (identik), Equipment_Health_Index = Equipment_Health_Score (identik),
#     Electrical_Exposure_Score ~ UPS_Load_Percent (korelasi 1,0), Employee_Risk_Index ~ Fatigue_Score (0,94),
#     Environmental_Risk_Index ~ Cooling_Utilization_Percent (0,89).
#     Apparent_Power_kVA, Active_Power_kW, Reactive_Power_kVAR dihitung dari Voltage, Current, Power_Factor
#     (Current vs Apparent_Power_kVA korelasi 1,0), sehingga bikin model linear tidak stabil (multikolinearitas).
#     Dibuang supaya model memakai variabel mentah yang bisa diubah langsung di simulasi Monte Carlo.
dup_cols = ["Fatigue_Index", "Equipment_Health_Index", "Electrical_Exposure_Score", "Employee_Risk_Index",
            "Environmental_Risk_Index", "Safety_Compliance_Index", "Training_Effectiveness_Score",
            "Apparent_Power_kVA", "Active_Power_kW", "Reactive_Power_kVAR"]

drop_cols = set(leak_cols + id_cols + date_cols + text_cols + const_cols + dup_cols)
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[TARGET]

cat_features = X.select_dtypes(exclude="number").columns.tolist()
num_features = X.select_dtypes("number").columns.tolist()
print(f"Fitur numerik: {len(num_features)}, fitur kategorik: {len(cat_features)}")
print("Fitur kategorik:", cat_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", X_train.shape, " Test:", X_test.shape)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

## 4. Melatih dan membandingkan model regresi

Empat model dibandingkan:
- **Linear Regression**: baseline paling sederhana dan mudah diinterpretasi.
- **Ridge Regression**: regresi linear dengan regularisasi (mencegah overfitting karena fiturnya banyak).
- **Random Forest**: kumpulan pohon keputusan, bisa menangkap pola non-linear.
- **Gradient Boosting** (HistGradientBoosting): pohon yang dibangun bertahap untuk memperbaiki kesalahan sebelumnya.

Metrik: **MAE** (rata-rata selisih absolut), **RMSE** (menghukum kesalahan besar), dan **R²** (proporsi variasi yang dijelaskan model, makin dekat 1 makin baik). Selain test set, dipakai juga **5-fold cross validation** pada data train.

**Formulasi Ridge Regression.** Skor risiko pekerjaan ke-$i$ dimodelkan sebagai kombinasi linear fitur yang sudah distandarkan:

$$\hat{y}_i = \beta_0 + \sum_{j=1}^{p} \beta_j x_{ij}$$

Koefisien $\beta$ dicari dengan meminimalkan galat kuadrat ditambah penalti L2 (Hoerl & Kennard, 1970):

$$\min_{\beta} \; \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} \beta_j^2, \qquad \alpha = 10$$

Penalti $\alpha$ membuat koefisien lebih stabil saat fiturnya banyak dan saling berkorelasi. Rumus metrik evaluasinya:

$$\text{MAE} = \frac{1}{n}\sum |y_i - \hat{y}_i|, \quad \text{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}, \quad R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
    "Gradient Boosting": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=SEED),
}

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
results, fitted = [], {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    results.append({
        "Model": name,
        "CV R2 (train)": cv_r2.mean(),
        "MAE (test)": mean_absolute_error(y_test, pred),
        "RMSE (test)": np.sqrt(mean_squared_error(y_test, pred)),
        "R2 (test)": r2_score(y_test, pred),
    })

results = pd.DataFrame(results).sort_values("R2 (test)", ascending=False).reset_index(drop=True)
results.round(3)

In [ ]:
best_name = results.loc[0, "Model"]
best_model = fitted[best_name]
y_pred = best_model.predict(X_test)
print("Model terbaik:", best_name)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(y_test, y_pred, alpha=0.35, s=12)
lims = [y.min(), y.max()]
ax[0].plot(lims, lims, "r--", label="prediksi = aktual")
ax[0].set_xlabel("Skor aktual"); ax[0].set_ylabel("Skor prediksi")
ax[0].set_title(f"Aktual vs Prediksi ({best_name})"); ax[0].legend()

residuals = y_test - y_pred
sns.histplot(residuals, bins=40, kde=True, ax=ax[1], color="gray")
ax[1].set_title("Distribusi residual (aktual − prediksi)")
plt.tight_layout(); plt.show()
print(f"Standar deviasi residual: {residuals.std():.2f} poin")

## 5. Faktor yang paling berpengaruh

Dipakai **permutation importance**: nilai satu fitur diacak, lalu dilihat seberapa besar R² model turun. Makin besar penurunannya, makin penting fitur tersebut. Cara ini berlaku untuk model apa pun.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=SEED, n_jobs=-1, scoring="r2")
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

top = imp.head(15)
plt.figure(figsize=(8, 5.5))
sns.barplot(x=top.values, y=top.index, color="steelblue")
plt.xlabel("Penurunan R² saat fitur diacak")
plt.title("15 fitur paling berpengaruh")
plt.tight_layout(); plt.show()
top.round(4)

## 6. Simulasi Monte Carlo

### 6.1 Skenario *what-if* (deterministik)

Model regresi hanya memprediksi skor untuk satu pekerjaan. Dengan **Monte Carlo**, kita mensimulasikan ribuan pekerjaan acak untuk melihat **distribusi** risiko dan **peluang** sebuah pekerjaan masuk kategori Medium/High pada berbagai skenario.

**Cara kerja satu kali simulasi (satu skenario):**
1. Ambil `N = 5.000` pekerjaan secara acak *dengan pengembalian* (bootstrap) dari data. Ini menjaga hubungan antar-variabel tetap realistis.
2. Ubah variabel sesuai skenario (misalnya `Night_Shift_Flag = 1` untuk semua).
3. Prediksi skor dengan model terbaik.
4. Tambahkan **noise acak** dari distribusi residual model, karena model tidak sempurna (ketidakpastian model ikut disimulasikan).
5. Hitung rata-rata skor, `P(skor ≥ 40)` (Medium ke atas), dan `P(skor > 65)` (High).

Secara matematis, skor simulasi untuk pekerjaan $k$ adalah $\tilde{y}_k = f(\mathbf{x}_k^{\text{skenario}}) + \varepsilon_k$, dengan $f$ model Ridge dan $\varepsilon_k$ diambil acak dari residual data test. Peluangnya diestimasi sebagai $\hat{P}(\tilde{y} \geq 40) = \frac{1}{N}\sum_{k=1}^{N} \mathbf{1}[\tilde{y}_k \geq 40]$.

Langkah 1–5 diulang `R = 200` kali per skenario, sehingga kita mendapat **rata-rata dan interval kepercayaan 95%** untuk setiap angka.

In [ ]:
N_SAMPLES = 5000   # pekerjaan per simulasi
N_RUNS = 200       # pengulangan per skenario
MEDIUM_T, HIGH_T = 40, 65
rng = np.random.default_rng(SEED)

# residual dari data test dipakai sebagai sumber noise ketidakpastian model
resid_pool = (y_test - best_model.predict(X_test)).values

def fatigue_down(d, pct=0.2):
    d["Fatigue_Score"] = d["Fatigue_Score"] * (1 - pct)
    return d

def equipment_better(d, pts=10):
    d["Equipment_Health_Score"] = np.clip(d["Equipment_Health_Score"] + pts, 0, 100)
    return d

scenarios = {
    "Baseline (kondisi sekarang)": lambda d: d,
    "Semua shift malam":           lambda d: d.assign(Night_Shift_Flag=1),
    "Tanpa shift malam":           lambda d: d.assign(Night_Shift_Flag=0),
    "Kelelahan turun 20%":         fatigue_down,
    "Kesehatan alat +10 poin":     equipment_better,
    "LOTO 100% patuh":             lambda d: d.assign(LOTO_Violation_Flag=0),
    "Kombinasi perbaikan":         lambda d: equipment_better(fatigue_down(d.assign(Night_Shift_Flag=0, LOTO_Violation_Flag=0))),
}

def run_scenario(modifier):
    stats = []
    for _ in range(N_RUNS):
        idx = rng.integers(0, len(X), N_SAMPLES)
        sample = modifier(X.iloc[idx].copy())
        score = best_model.predict(sample) + rng.choice(resid_pool, N_SAMPLES)
        score = np.clip(score, 0, 100)
        stats.append((score.mean(), (score >= MEDIUM_T).mean(), (score > HIGH_T).mean()))
    return np.array(stats), score   # score terakhir disimpan untuk plot distribusi

mc_stats, mc_last = {}, {}
for name, mod in scenarios.items():
    mc_stats[name], mc_last[name] = run_scenario(mod)
print("Simulasi selesai:", len(scenarios), "skenario x", N_RUNS, "run x", N_SAMPLES, "pekerjaan")

In [ ]:
def ci(a):
    return f"{np.mean(a):.2f} [{np.percentile(a, 2.5):.2f} – {np.percentile(a, 97.5):.2f}]"

def ci_pct(a):
    return f"{100*np.mean(a):.1f}% [{100*np.percentile(a, 2.5):.1f} – {100*np.percentile(a, 97.5):.1f}]"

base_mean = mc_stats["Baseline (kondisi sekarang)"][:, 0].mean()
summary = pd.DataFrame({
    name: {
        "Rata-rata skor [CI 95%]": ci(s[:, 0]),
        "Perubahan vs baseline": f"{s[:, 0].mean() - base_mean:+.2f}",
        "P(Medium/High, skor ≥ 40)": ci_pct(s[:, 1]),
        "P(High, skor > 65)": ci_pct(s[:, 2]),
    } for name, s in mc_stats.items()
}).T
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name in ["Baseline (kondisi sekarang)", "Semua shift malam", "Kombinasi perbaikan"]:
    sns.kdeplot(mc_last[name], ax=ax[0], label=name, fill=True, alpha=0.25)
ax[0].axvline(MEDIUM_T, color="orange", ls="--"); ax[0].axvline(HIGH_T, color="red", ls="--")
ax[0].set_xlabel("Skor risiko simulasi"); ax[0].set_title("Distribusi skor hasil Monte Carlo")
ax[0].legend(fontsize=8)

names = list(mc_stats.keys())
p_med = [100 * mc_stats[n][:, 1].mean() for n in names]
err = [[100 * (mc_stats[n][:, 1].mean() - np.percentile(mc_stats[n][:, 1], 2.5)) for n in names],
       [100 * (np.percentile(mc_stats[n][:, 1], 97.5) - mc_stats[n][:, 1].mean()) for n in names]]
ax[1].barh(names, p_med, xerr=err, color="steelblue", capsize=3)
ax[1].invert_yaxis()
ax[1].set_xlabel("% pekerjaan dengan skor ≥ 40 (Medium/High)")
ax[1].set_title("Peluang risiko Medium/High per skenario (CI 95%)")
plt.tight_layout(); plt.show()

### 6.2 Analisis ketidakpastian probabilistik + tornado chart

Skenario di 6.1 memakai perubahan yang **pasti** (misalnya kelelahan turun tepat 20%). Di dunia nyata, kondisi operasi tidak pasti, jadi di sini setiap kondisi diperlakukan sebagai **variabel acak** dengan distribusi tertentu. Distribusi segitiga dipakai karena cukup ditentukan oleh nilai minimum, paling mungkin (*mode*), dan maksimum, seperti yang lazim di analisis risiko kuantitatif.

| Variabel ketidakpastian | Distribusi | Min | Mode | Maks | Dasar penentuan |
|---|---|---|---|---|---|
| Proporsi pekerjaan shift malam | Segitiga | 0,20 | 0,46 | 0,70 | Proporsi di data = 0,46 |
| Proporsi kerja akhir pekan | Segitiga | 0,10 | 0,29 | 0,50 | Proporsi di data = 0,29 |
| Perubahan skor kelelahan (%) | Segitiga | −30 | 0 | +20 | Program manajemen kelelahan vs lembur tinggi |
| Perubahan kesehatan alat (poin) | Segitiga | −10 | 0 | +10 | Perawatan tertunda vs perawatan intensif |
| Tingkat pelanggaran LOTO | Uniform | 0,00 | – | 0,10 | Di data = 0,043 |
| Perubahan beban UPS (poin %) | Uniform | −10 | – | +10 | Variasi beban operasi |

**Algoritma (diulang `N_ITER = 1.000` kali):**
1. Ambil satu nilai acak untuk setiap variabel di tabel.
2. Bootstrap 2.000 pekerjaan dari data, lalu terapkan kondisi tersebut. Shift malam, akhir pekan, dan pelanggaran LOTO diundi per pekerjaan dengan distribusi Bernoulli sesuai proporsinya.
3. Prediksi skor dengan model Ridge, lalu tambahkan noise residual.
4. Simpan rata-rata skor, $P(\text{Medium/High})$, dan $P(\text{High})$.

Hasilnya adalah **distribusi** peluang risiko (P5, P50, P95). Lalu **korelasi peringkat Spearman** antara setiap variabel input dan $P(\text{Medium/High})$ dipakai untuk membuat **tornado chart**, yaitu urutan variabel yang paling memengaruhi ketidakpastian risiko.

In [ ]:
N_ITER = 1000
N_JOBS_PER_ITER = 2000
rng_mc = np.random.default_rng(SEED + 1)

def sample_inputs(rng):
    return {
        "Proporsi shift malam":         rng.triangular(0.20, 0.46, 0.70),
        "Proporsi kerja akhir pekan":   rng.triangular(0.10, 0.29, 0.50),
        "Perubahan kelelahan (%)":      rng.triangular(-30, 0, 20),
        "Perubahan kesehatan alat":     rng.triangular(-10, 0, 10),
        "Tingkat pelanggaran LOTO":     rng.uniform(0.00, 0.10),
        "Perubahan beban UPS":          rng.uniform(-10, 10),
    }

def apply_inputs(d, u, rng):
    n = len(d)
    d["Night_Shift_Flag"] = (rng.random(n) < u["Proporsi shift malam"]).astype(int)
    d["Weekend_Work_Flag"] = (rng.random(n) < u["Proporsi kerja akhir pekan"]).astype(int)
    d["Fatigue_Score"] = np.clip(d["Fatigue_Score"] * (1 + u["Perubahan kelelahan (%)"] / 100), 0, 100)
    d["Equipment_Health_Score"] = np.clip(d["Equipment_Health_Score"] + u["Perubahan kesehatan alat"], 0, 100)
    d["LOTO_Violation_Flag"] = (rng.random(n) < u["Tingkat pelanggaran LOTO"]).astype(int)
    d["UPS_Load_Percent"] = np.clip(d["UPS_Load_Percent"] + u["Perubahan beban UPS"], 0, 100)
    return d

records = []
for it in range(N_ITER):
    u = sample_inputs(rng_mc)
    idx = rng_mc.integers(0, len(X), N_JOBS_PER_ITER)
    sample = apply_inputs(X.iloc[idx].copy(), u, rng_mc)
    score = np.clip(best_model.predict(sample) + rng_mc.choice(resid_pool, N_JOBS_PER_ITER), 0, 100)
    records.append({**u,
                    "Rata-rata skor": score.mean(),
                    "P(Medium/High) %": 100 * (score >= MEDIUM_T).mean(),
                    "P(High) %": 100 * (score > HIGH_T).mean()})
    if (it + 1) % 250 == 0:
        print(f"Selesai {it + 1}/{N_ITER} iterasi")

df_mc = pd.DataFrame(records)
outputs = ["Rata-rata skor", "P(Medium/High) %", "P(High) %"]
stats_mc = pd.DataFrame({
    "Mean": df_mc[outputs].mean(),
    "P5": df_mc[outputs].quantile(0.05),
    "P50 (median)": df_mc[outputs].quantile(0.50),
    "P95": df_mc[outputs].quantile(0.95),
}).round(2)
stats_mc

In [ ]:
input_cols = list(sample_inputs(np.random.default_rng(0)).keys())
rho = pd.Series({c: spearmanr(df_mc[c], df_mc["P(Medium/High) %"])[0] for c in input_cols})
rho = rho.reindex(rho.abs().sort_values().index)

p_over_50 = (df_mc["P(Medium/High) %"] >= 50).mean() * 100
print(f"Peluang lebih dari separuh pekerjaan berisiko Medium/High: {p_over_50:.1f}% dari skenario")

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
vals = df_mc["P(Medium/High) %"]
ax[0].hist(vals, bins=35, color="#2b8cbe", edgecolor="white", alpha=0.8)
for q, ls in [(0.05, ":"), (0.5, "-"), (0.95, ":")]:
    ax[0].axvline(vals.quantile(q), color="black", ls=ls, lw=1.3, label=f"P{int(q*100)} = {vals.quantile(q):.1f}%")
ax[0].set_xlabel("% pekerjaan berisiko Medium/High (skor ≥ 40)")
ax[0].set_ylabel("Jumlah iterasi")
ax[0].set_title(f"Distribusi peluang risiko Medium/High (N = {N_ITER})")
ax[0].legend()

colors = ["#d62728" if v > 0 else "#2ca02c" for v in rho.values]
ax[1].barh(rho.index, rho.values, color=colors)
ax[1].axvline(0, color="black", lw=0.8)
ax[1].set_xlabel("Korelasi Spearman dengan P(Medium/High)")
ax[1].set_title("Tornado chart: sumber ketidakpastian risiko")
for i, v in enumerate(rho.values):
    ax[1].text(v + (0.02 if v >= 0 else -0.02), i, f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=9)
ax[1].set_xlim(-1.1, 1.1)
plt.tight_layout(); plt.show()
rho.sort_values(key=abs, ascending=False).round(3)

## 7. Kesimpulan

*(Angka di bawah dari hasil run dengan `SEED = 42`; kalau dijalankan ulang hasilnya sama atau beda sangat tipis.)*

- **Model regresi:** Ridge Regression dan Linear Regression jadi model terbaik dengan **R² ≈ 0,77** dan **MAE ≈ 4 poin** di data test, sedikit lebih baik dari Gradient Boosting (≈ 0,76) dan Random Forest (0,74). Artinya hubungan faktor risiko dengan skor di dataset ini sebagian besar linear, jadi model sederhana sudah cukup dan lebih mudah dijelaskan.
- **Faktor paling berpengaruh:** shift malam, kesehatan peralatan, tingkat kelelahan pekerja, pelanggaran LOTO, kerja di akhir pekan, dan beban UPS.
- **Simulasi Monte Carlo** (7 skenario × 200 run × 5.000 pekerjaan):
  - Kondisi sekarang: rata-rata skor ≈ 39,8 dan sekitar **48%** pekerjaan berisiko Medium/High.
  - Kalau semua pekerjaan dilakukan saat shift malam, angkanya naik ke sekitar **71%**. Kalau shift malam dihindari, turun ke sekitar **29%**.
  - Kelelahan turun 20% menurunkan peluang Medium/High ke sekitar **37%**, kesehatan alat +10 poin ke sekitar **39%**, sedangkan LOTO 100% patuh efeknya kecil (sekitar **46%**) karena pelanggaran LOTO di data cuma 4%.
  - **Kombinasi perbaikan** (tanpa shift malam, kelelahan turun, alat lebih sehat, LOTO patuh) menurunkan peluang Medium/High ke sekitar **9%** dan risiko High hampir nol.
- Interval kepercayaan 95% yang sempit menunjukkan hasil simulasi stabil, dan ketidakpastian model (residual) sudah ikut dihitung.
- **Monte Carlo probabilistik** (1.000 iterasi, 6 variabel acak): kalau kondisi operasi tidak pasti, peluang pekerjaan berisiko Medium/High berada di rentang **31% (P5) – 63% (P95)**, dengan median **47%**. Pada sekitar **37%** iterasi, lebih dari separuh pekerjaan masuk kategori Medium/High.
- **Tornado chart** menunjukkan sumber ketidakpastian terbesar adalah **perubahan kelelahan pekerja** (ρ = +0,63), disusul **proporsi shift malam** (+0,44), **kesehatan alat** (−0,43, makin sehat makin aman), dan **beban UPS** (+0,38). Jadi program **manajemen kelelahan** dan **pengaturan shift** adalah pengendalian yang paling efektif untuk menekan risiko.

**Keterbatasan:**
- Dataset kemungkinan besar sintetis, jadi angka-angka di sini menggambarkan pola dalam data, bukan bukti kausal di data center nyata.
- Skenario mengubah satu variabel tanpa mengubah variabel lain yang mungkin ikut berubah di dunia nyata (misalnya shift malam bisa ikut menaikkan kelelahan).
- Batas kategori (40 dan 65) diambil dari kolom `Risk_Level` di dataset.
- Rentang distribusi di 6.2 adalah asumsi penulis berdasarkan nilai di data, bukan hasil pengukuran lapangan.

## 8. Referensi

1. Dataset: *Electrical Safety* (`electrical_safety_5000.csv`), Kaggle.
2. Hoerl, A. E., & Kennard, R. W. (1970). Ridge regression: Biased estimation for nonorthogonal problems. *Technometrics*, 12(1), 55–67.
3. Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32.
4. Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
5. Metropolis, N., & Ulam, S. (1949). The Monte Carlo method. *Journal of the American Statistical Association*, 44(247), 335–341.
6. Efron, B. (1979). Bootstrap methods: Another look at the jackknife. *The Annals of Statistics*, 7(1), 1–26.
7. Spearman, C. (1904). The proof and measurement of association between two things. *The American Journal of Psychology*, 15(1), 72–101.
8. NFPA 70E (2024). *Standard for Electrical Safety in the Workplace*. National Fire Protection Association.
9. OSHA 29 CFR 1910.147. *The Control of Hazardous Energy (Lockout/Tagout)*. U.S. Department of Labor.